# Phi-4-reasoning-vision-15B evaluation notebook — 12 combinations

This notebook is a Phi-4-specific evaluation notebook adapted from the uploaded
retrieval-augmented report-generation workflow.

## What it does
- loads **Phi-4-reasoning-vision-15B** and an optional fine-tuned **LoRA** adapter
- uses **image + text** as model input for every generation pass
- retrieves top-K reference cases from a BioMedCLIP/FAISS gallery
- generates a **final report text** only (no JSON required)
- optionally runs **confession / self-revision passes**, where **each pass re-prompts
  Phi-4 with the same query image** plus updated textual context
- evaluates **only the final output report text** against the ground-truth report text

## Experiment grid — 3 datasets × 4 methods = 12 combinations

| Method | USE_LORA | USE_CONFESSION | Behaviour |
|---|---|---|---|
| `phi4_reason_Noverify_Noconfess` | False | False | Single-pass baseline |
| `phi4_reason_base_confess` | **False** | **True** | **4-pass confession, no LoRA verifier** ← new |
| `phi4_reason_verify_Noconfess` | True | False | Pass 1 + LoRA verify score |
| `phi4_reason_verify_confess` | True | True | Up-to-4 passes, verify-gated exits |

The `base_confess` method runs all 4 confession passes unconditionally — there are
no LoRA-gated early exits, making the loop fully deterministic.

## Main outputs
- per-case JSONL with retrieved evidence, intermediate drafts, and final report text
- summary CSV with text-generation metrics
- combined 12-combo comparison CSV


In [1]:
# ============================================================
# 0) Configuration
# ============================================================
from __future__ import annotations

from pathlib import Path
import os

# -------- Phi-4 model + LoRA --------
MODEL_ID  = "microsoft/Phi-4-reasoning-vision-15B"
LORA_DIR  = Path("/data/liangz2/openi/Phi_4_reason/phi4_lora")
HF_TOKEN_PATH = Path("/data/liangz2/openi/hf_token.txt")

# -------- Dataset roots — all three active --------
DATASET_ROOTS = {
    "mimic"   : Path("/data/liangz2/openi/faiss_val_mimic_biomedclip"),
    "openi"   : Path("/data/liangz2/openi/faiss_val_openi_biomedclip"),
    "combined": Path("/data/liangz2/openi/faiss_val_combined_biomedclip"),
}

def infer_gallery_root(val_root: Path) -> Path:
    return Path(str(val_root).replace("faiss_val_", "faiss_train_", 1))

# -------- 4 experiment methods (3 datasets × 4 = 12 combinations) --------
EXPERIMENTS = [
    {
        "name": "phi4_reason_Noverify_Noconfess",
        "USE_LORA": False,
        "USE_CONFESSION": False,
        "subdir": "phi4_reason_Noverify_Noconfess",
    },
    {
        # NEW: base model with full 4-pass confession loop — no LoRA verifier.
        # All 4 passes run unconditionally; no verify-gated early exits.
        "name": "phi4_reason_base_confess",
        "USE_LORA": False,
        "USE_CONFESSION": True,
        "subdir": "phi4_reason_base_confess",
    },
    {
        "name": "phi4_reason_verify_Noconfess",
        "USE_LORA": True,
        "USE_CONFESSION": False,
        "subdir": "phi4_reason_verify_Noconfess",
    },
    {
        "name": "phi4_reason_verify_confess",
        "USE_LORA": True,
        "USE_CONFESSION": True,
        "subdir": "phi4_reason_verify_confess",
    },
]

# -------- Runtime --------
DEVICE = "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES", None) is not None or os.path.exists("/dev/nvidia0") else "cpu"
DTYPE  = "bfloat16"
TOP_K  = 10
MAX_NEW_TOKENS = 220
DO_SAMPLE   = False
TEMPERATURE = 0.2
TOP_P       = 0.95

# -------- Confession / iterative self-revision --------
MAX_CONFESSION_PASSES = 4   # pass1 initial, pass2 majority, pass3 minority, pass4 mixed final
STOP_IF_UNCHANGED = True

# -------- Batch eval --------
N_ITEMS        = None          # None -> full dataset
PROGRESS_EVERY = 50

# -------- Outputs --------
SAVE_ROOT = Path("/data/liangz2/openi/phi4_rag_eval_12combo")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = SAVE_ROOT / "phi4_12combo_comparison.csv"

print("MODEL_ID:", MODEL_ID)
print("LORA_DIR:", LORA_DIR)
print("SAVE_ROOT:", SAVE_ROOT)
print(f"Experiments: {len(EXPERIMENTS)} methods × {len(DATASET_ROOTS)} datasets = "
      f"{len(EXPERIMENTS)*len(DATASET_ROOTS)} combinations")
print("DATASET_ROOTS:")
for k, v in DATASET_ROOTS.items():
    print(" ", k, "->", v, "| gallery:", infer_gallery_root(v))


MODEL_ID: microsoft/Phi-4-reasoning-vision-15B
LORA_DIR: /data/liangz2/openi/Phi_4_reason/phi4_lora
SAVE_ROOT: /data/liangz2/openi/phi4_rag_eval_9combo
DATASET_ROOTS:
  mimic -> /data/liangz2/openi/faiss_val_mimic_biomedclip | gallery: /data/liangz2/openi/faiss_train_mimic_biomedclip
  openi -> /data/liangz2/openi/faiss_val_openi_biomedclip | gallery: /data/liangz2/openi/faiss_train_openi_biomedclip
  combined -> /data/liangz2/openi/faiss_val_combined_biomedclip | gallery: /data/liangz2/openi/faiss_train_combined_biomedclip


In [2]:

# ============================================================
# 1) Login + imports
# ============================================================
from huggingface_hub import login

if HF_TOKEN_PATH.exists():
    with open(HF_TOKEN_PATH, "r", encoding="utf-8") as f:
        hf_token = f.readline().strip()
    if hf_token:
        login(token=hf_token)
        print("✅ Hugging Face login successful.")
    else:
        print("⚠️ HF token file is empty.")
else:
    print(f"⚠️ HF token file not found: {HF_TOKEN_PATH}")

import json
import csv
import math
import re
import time
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter

import numpy as np
from PIL import Image

import torch
import faiss
import open_clip
from peft import PeftModel
from transformers import AutoProcessor, AutoConfig, AutoTokenizer
from transformers.dynamic_module_utils import get_class_from_dynamic_module


✅ Hugging Face login successful.


In [3]:
# ============================================================
# 2) Load Phi-4-reasoning-vision-15B backbone + LoRA wrapper
# ============================================================
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if getattr(processor, "tokenizer", None) is not None and processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

Phi4ForCausalLMV = get_class_from_dynamic_module(
    class_reference="modeling_phi4_visionr.Phi4ForCausalLMV",
    pretrained_model_name_or_path=MODEL_ID,
)

backbone = Phi4ForCausalLMV.from_pretrained(
    MODEL_ID,
    config=config,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

if not LORA_DIR.exists():
    raise FileNotFoundError(f"LORA_DIR not found: {LORA_DIR}")

verifier_base = PeftModel.from_pretrained(backbone, str(LORA_DIR))
verifier_base.eval()

print("✅ Phi-4 backbone + LoRA wrapper loaded")
print("Backbone type:", type(backbone))
print("Verifier wrapper type:", type(verifier_base))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Phi-4 backbone + LoRA wrapper loaded
Backbone type: <class 'transformers_modules.microsoft.Phi_hyphen_4_hyphen_reasoning_hyphen_vision_hyphen_15B.8e0aeaf8f5c28e5784c123e94c85fd1a919704f8.modeling_phi4_visionr.Phi4ForCausalLMV'>
Verifier wrapper type: <class 'peft.peft_model.PeftModelForCausalLM'>


In [4]:
# ============================================================
# 3) Phi-4 multimodal prompt helpers + LoRA on/off utilities
# ============================================================
PHI4_SYSTEM = (
    "You are a radiology assistant for chest X-ray interpretation. "
    "Use the query image and retrieved reference evidence to generate a concise final chest X-ray report. "
    "Output only the final report text. Do not output JSON, bullets, markdown, or extra commentary."
)


def render_phi4_chat(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool) -> str:
    msgs = messages[:]
    if len(msgs) == 0 or msgs[0].get("role") != "system":
        msgs = [{"role": "system", "content": PHI4_SYSTEM}] + msgs

    text = tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        return_dict=False,
    )

    # Phi-4 non-thinking generation token used in prior experiments
    if add_generation_prompt:
        text = str(text) + "<|dummy_84|>"
    return text


def ensure_image_placeholder(text: str) -> str:
    markers = ["<image>", "<|image_1|>", "<|image|>", "<image_1>", "<start_of_image>"]
    if any(m in text for m in markers):
        return text
    return "<image>\n" + text


# ----------------------------
# Utilities: adapter toggling
# ----------------------------
def lora_off(model: PeftModel):
    """
    Disable LoRA adapters (model behaves like base).
    Compatible with PEFT >= 0.10 where disable_adapter() exists.
    """
    try:
        model.disable_adapter()
    except Exception:
        model.set_adapter([])


def lora_on(model: PeftModel):
    """
    Enable LoRA adapters (model behaves like LoRA-tuned).
    """
    try:
        model.enable_adapter()
    except Exception:
        model.set_adapter(model.active_adapter or "default")


# ----------------------------
# Generation pass (LoRA OFF)
# ----------------------------
@torch.inference_mode()
def generate_report_base(
    model: PeftModel,
    processor,
    image_path: str,
    prompt_messages: List[Dict[str, str]],
    max_new_tokens: int = MAX_NEW_TOKENS,
    do_sample: bool = DO_SAMPLE,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
) -> str:
    lora_off(model)

    image = Image.open(image_path).convert("RGB")
    prompt_text = render_phi4_chat(
        processor.tokenizer,
        prompt_messages,
        add_generation_prompt=True,
    )
    prompt_text = ensure_image_placeholder(prompt_text)

    enc = processor(
        text=prompt_text,
        images=[image],
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    model_inputs = {}
    for k, v in enc.items():
        if torch.is_tensor(v):
            model_inputs[k] = v.to(model.device)
        else:
            model_inputs[k] = v

    out_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

    gen_ids = out_ids[0][model_inputs["input_ids"].shape[1]:]
    text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text.strip()


# -----------------------------------------------------------------------------------
# Verification pass (LoRA ON)
#   Outputs True/False probability using logits of tokens "TRUE"/"FALSE"
#   Assumes your LoRA classifier was trained to answer exactly TRUE or FALSE.
# -----------------------------------------------------------------------------------
@torch.inference_mode()
def verify_true_false_lora(
    model: PeftModel,
    tokenizer: AutoTokenizer,
    verify_prompt: str,
) -> dict:
    lora_on(model)

    messages = [{"role": "user", "content": verify_prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    logits = model(input_ids=input_ids).logits
    next_logits = logits[:, -1, :]

    true_ids = tokenizer.encode("TRUE", add_special_tokens=False)
    false_ids = tokenizer.encode("FALSE", add_special_tokens=False)

    if len(true_ids) == 1 and len(false_ids) == 1:
        tid, fid = true_ids[0], false_ids[0]
        two = torch.stack([next_logits[0, tid], next_logits[0, fid]], dim=0)
        probs = torch.softmax(two, dim=0)
        p_true = float(probs[0].item())
        p_false = float(probs[1].item())
        pred = "TRUE" if p_true >= p_false else "FALSE"
        return {"pred": pred, "p_true": p_true, "p_false": p_false}

    gen = model.generate(
        input_ids=input_ids,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    ans = tokenizer.decode(gen[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    ans_up = ans.upper()
    if "TRUE" in ans_up and "FALSE" not in ans_up:
        return {"pred": "TRUE", "p_true": 1.0, "p_false": 0.0}
    if "FALSE" in ans_up and "TRUE" not in ans_up:
        return {"pred": "FALSE", "p_true": 0.0, "p_false": 1.0}
    return {"pred": "UNKNOWN", "p_true": float("nan"), "p_false": float("nan")}

In [5]:

# ============================================================
# 4) Load BioMedCLIP for FAISS query embeddings
# ============================================================
BIOCLIP_MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

biomedclip_model, biomedclip_preprocess = open_clip.create_model_from_pretrained(BIOCLIP_MODEL_ID)
biomedclip_tokenizer = open_clip.get_tokenizer(BIOCLIP_MODEL_ID)
biomedclip_model = biomedclip_model.to(verifier_base.device).eval()

@torch.inference_mode()
def biomedclip_image_embedding(image_path: str, l2_normalize: bool = True) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    x = biomedclip_preprocess(img).unsqueeze(0).to(model.device)
    feat = biomedclip_model.encode_image(x)
    feat = feat.float()
    if l2_normalize:
        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return feat[0].detach().cpu().numpy().astype(np.float32)


In [6]:

# ============================================================
# 5) FAISS helpers
# ============================================================

def load_faiss_bundle(faiss_dir: str | Path) -> Tuple[faiss.Index, List[Dict[str, Any]]]:
    faiss_dir = Path(faiss_dir)
    index_path = faiss_dir / "faiss_image.index"
    meta_path = faiss_dir / "metadata.jsonl"
    if not index_path.exists():
        raise FileNotFoundError(f"Missing: {index_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing: {meta_path}")

    index = faiss.read_index(str(index_path))
    meta = []
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                meta.append(json.loads(line))
    return index, meta


def load_dataset_bundles(dataset_root: str | Path) -> Tuple[faiss.Index, List[Dict[str, Any]], faiss.Index, List[Dict[str, Any]]]:
    dataset_root = Path(dataset_root)
    gallery_root = infer_gallery_root(dataset_root)
    test_index, test_meta = load_faiss_bundle(dataset_root)
    gal_index, gal_meta = load_faiss_bundle(gallery_root)
    print(f"✅ Loaded dataset bundles for {dataset_root.name}")
    print("test size:", len(test_meta), "gallery size:", len(gal_meta))
    return test_index, test_meta, gal_index, gal_meta


def query_vec_from_test_id(test_index: faiss.Index, test_id: int) -> np.ndarray:
    vec = test_index.reconstruct(int(test_id))
    vec = np.asarray(vec, dtype=np.float32)
    if vec.ndim == 1:
        vec = vec[None, :]
    return vec


def retrieve_gallery_neighbors(gallery_index: faiss.Index, gallery_meta: List[Dict[str, Any]], qvec: np.ndarray, k: int = 10) -> List[Dict[str, Any]]:
    D, I = gallery_index.search(qvec.astype(np.float32), int(k))
    neighbors: List[Dict[str, Any]] = []
    for rank, (idx, dist) in enumerate(zip(I[0].tolist(), D[0].tolist()), start=1):
        if idx < 0 or idx >= len(gallery_meta):
            continue
        row = dict(gallery_meta[idx])
        row["_rank"] = rank
        row["_score"] = float(dist)
        neighbors.append(row)
    return neighbors


In [7]:
# ============================================================
# 6) Evidence formatting + target report helpers
# ============================================================
NO_DETAIL = "No detailed information"


def _safe_str(x: Any) -> str:
    return (str(x) if x is not None else "").strip()


def _row_caption(row: Dict[str, Any]) -> str:
    for k in ["caption", "pred_caption", "report", "report_text"]:
        v = _safe_str(row.get(k, ""))
        if v:
            return v
    return NO_DETAIL


def _row_findings(row: Dict[str, Any]) -> str:
    for k in ["FINDINGS", "findings"]:
        v = _safe_str(row.get(k, ""))
        if v:
            return v
    return NO_DETAIL


def _row_impression(row: Dict[str, Any]) -> str:
    for k in ["IMPRESSION", "impression"]:
        v = _safe_str(row.get(k, ""))
        if v:
            return v
    return NO_DETAIL


def _row_normal(row: Dict[str, Any]) -> str:
    v = _safe_str(row.get("normal", "")).lower()
    return v if v in {"yes", "no"} else "unknown"


def _row_labels(row: Dict[str, Any]) -> List[str]:
    labs = row.get("labels", [])
    if isinstance(labs, list):
        return [str(x).strip().lower() for x in labs if str(x).strip()]
    return []


def build_reference_block(neighbors: List[Dict[str, Any]], k: int) -> str:
    top = neighbors[: max(1, min(int(k), len(neighbors)))]
    blocks = []
    for i, row in enumerate(top, start=1):
        blocks.append(
            f"Reference {i}:\n"
            f"normal: {_row_normal(row)}\n"
            f"labels: {_row_labels(row)}\n"
            f"caption: {_row_caption(row)}\n"
            f"FINDINGS: {_row_findings(row)}\n"
            f"IMPRESSION: {_row_impression(row)}"
        )
    return "\n\n".join(blocks)


def build_gt_report_text(row: Dict[str, Any]) -> str:
    findings = _row_findings(row)
    impression = _row_impression(row)
    caption = _row_caption(row)

    parts = []
    if findings and findings != NO_DETAIL:
        parts.append(f"FINDINGS: {findings}")
    if impression and impression != NO_DETAIL:
        parts.append(f"IMPRESSION: {impression}")
    if not parts and caption and caption != NO_DETAIL:
        parts.append(caption)
    return "\n".join(parts).strip()

In [8]:

# ============================================================
# 7) Prompt builders for initial pass + verifier-guided confession
# ============================================================

def split_majority_minority(neighbors: List[Dict[str, Any]], k: int = 10) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], str]:
    top = neighbors[: max(1, min(int(k), len(neighbors)))]
    yes_rows = [r for r in top if _row_normal(r) == "yes"]
    no_rows = [r for r in top if _row_normal(r) == "no"]

    if len(yes_rows) >= len(no_rows):
        majority_flag = "yes"
        majority_rows = yes_rows if yes_rows else top
        minority_rows = no_rows
    else:
        majority_flag = "no"
        majority_rows = no_rows if no_rows else top
        minority_rows = yes_rows

    return majority_rows, minority_rows, majority_flag


def build_reference_block_with_tag(neighbors: List[Dict[str, Any]], tag: str, k: Optional[int] = None) -> str:
    use_rows = neighbors if k is None else neighbors[: max(1, min(int(k), len(neighbors)))]
    blocks = []
    for i, row in enumerate(use_rows, start=1):
        blocks.append(
            f"{tag} Reference {i}:\n"
            f"normal: {_row_normal(row)}\n"
            f"labels: {_row_labels(row)}\n"
            f"caption: {_row_caption(row)}\n"
            f"FINDINGS: {_row_findings(row)}\n"
            f"IMPRESSION: {_row_impression(row)}"
        )
    return "\n\n".join(blocks)


def build_initial_messages(neighbors: List[Dict[str, Any]], k: int = 10) -> List[Dict[str, str]]:
    evidence = build_reference_block(neighbors, k=k)
    user = (
        "Use the QUERY chest X-ray image together with the retrieved reference cases below to write the final report for the QUERY image.\n\n"
        "Return only the final report text with concise FINDINGS and IMPRESSION.\n\n"
        f"{evidence}"
    )
    return [
        {"role": "system", "content": PHI4_SYSTEM},
        {"role": "user", "content": user},
    ]


def build_majority_messages(majority_neighbors: List[Dict[str, Any]], prev_report: str, pass_id: int) -> List[Dict[str, str]]:
    evidence = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY", k=len(majority_neighbors))
    user = (
        f"This is confession pass {pass_id} for the same QUERY chest X-ray image.\n\n"
        "Revise the previous draft using ONLY the majority retrieved evidence below.\n"
        "Treat these majority references as the currently most reliable guidance.\n"
        "Return only the revised final report text.\n\n"
        f"Previous draft:\n{prev_report}\n\n"
        f"Majority retrieved evidence:\n{evidence}"
    )
    return [
        {"role": "system", "content": PHI4_SYSTEM},
        {"role": "user", "content": user},
    ]


def build_minority_false_messages(minority_neighbors: List[Dict[str, Any]], prev_report: str, pass_id: int) -> List[Dict[str, str]]:
    evidence = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors))
    user = (
        f"This is confession pass {pass_id} for the same QUERY chest X-ray image.\n\n"
        "The minority retrieved evidence below should be treated as likely FALSE statements or misleading evidence for the QUERY image.\n"
        "Use them only to avoid copying their false claims into the report.\n"
        "Revise the previous draft accordingly and return only the revised final report text.\n\n"
        f"Previous draft:\n{prev_report}\n\n"
        f"Minority retrieved evidence annotated as FALSE:\n{evidence}"
    )
    return [
        {"role": "system", "content": PHI4_SYSTEM},
        {"role": "user", "content": user},
    ]


def build_mixed_final_messages(
    majority_neighbors: List[Dict[str, Any]],
    minority_neighbors: List[Dict[str, Any]],
    prev_report: str,
    pass_id: int,
) -> List[Dict[str, str]]:
    majority_text = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY_TRUE", k=len(majority_neighbors))
    minority_text = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
    user = (
        f"This is the final confession pass {pass_id} for the same QUERY chest X-ray image.\n\n"
        "Use the MAJORITY_TRUE evidence as reliable support.\n"
        "Use the MINORITY_FALSE evidence as likely misleading or false statements to avoid.\n"
        "Revise the previous draft one final time and return only the final report text.\n\n"
        f"Previous draft:\n{prev_report}\n\n"
        f"MAJORITY_TRUE evidence:\n{majority_text}\n\n"
        f"MINORITY_FALSE evidence:\n{minority_text}"
    )
    return [
        {"role": "system", "content": PHI4_SYSTEM},
        {"role": "user", "content": user},
    ]


def build_verify_prompt(report_text: str, neighbors: List[Dict[str, Any]], mode: str = "majority") -> str:
    majority_neighbors, minority_neighbors, majority_flag = split_majority_minority(neighbors, k=len(neighbors))
    if mode == "majority":
        evidence = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY", k=len(majority_neighbors))
        instruction = (
            "Decide whether the draft report is supported by the majority retrieved evidence. "
            "Answer exactly TRUE or FALSE."
        )
    elif mode == "minority_false":
        evidence = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
        instruction = (
            "Decide whether the draft report avoids the likely false minority evidence below. "
            "Answer exactly TRUE or FALSE."
        )
    else:
        majority_text = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY_TRUE", k=len(majority_neighbors))
        minority_text = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
        evidence = f"{majority_text}\n\n{minority_text}"
        instruction = (
            "Decide whether the draft report is supported by the majority evidence and avoids the minority false evidence. "
            "Answer exactly TRUE or FALSE."
        )

    return (
        f"{instruction}\n\n"
        f"Draft report:\n{report_text}\n\n"
        f"Evidence:\n{evidence}"
    )


def normalize_report_text(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


In [9]:

# ============================================================
# 8) Phi-4 RAG report generation with verifier-guided confession
#   - Generation always runs with LoRA OFF (base backbone behaviour)
#   - Verification runs with LoRA ON only when USE_LORA is enabled
#   - The same query image is re-used on every confession pass
#
#   Behaviour by (use_lora, confession):
#     (False, False) -> 1 pass                              [Noverify_Noconfess]
#     (False, True)  -> 4 passes, no verify gating          [base_confess]  ← NEW
#     (True,  False) -> 1 pass + verify score stored        [verify_Noconfess]
#     (True,  True)  -> up-to-4 passes, verify-gated exits  [verify_confess]
# ============================================================

def rag_generate_phi4_report(
    *,
    model: PeftModel,
    processor,
    image_path: str,
    neighbors: List[Dict[str, Any]],
    k: int = 10,
    use_lora: bool = True,
    confession: bool = False,
) -> Dict[str, Any]:
    history: List[Dict[str, Any]] = []
    majority_neighbors, minority_neighbors, majority_flag = split_majority_minority(neighbors, k=k)

    # Pass 1: initial generation using full top-k retrieval
    messages = build_initial_messages(neighbors, k=k)
    report_1 = generate_report_base(
        model=model,
        processor=processor,
        image_path=image_path,
        prompt_messages=messages,
    )

    verify_1 = None
    if use_lora:
        verify_1 = verify_true_false_lora(
            model=model,
            tokenizer=processor.tokenizer,
            verify_prompt=build_verify_prompt(report_1, neighbors, mode="majority"),
        )

    history.append({
        "pass": 1,
        "stage": "initial_all_topk",
        "report": report_1,
        "verify": verify_1,
    })

    # No verifier or no confession -> stop early
    if not use_lora and not confession:
        return {
            "final_report": report_1,
            "history": history,
            "confession_enabled": bool(confession),
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    if use_lora and not confession:
        return {
            "final_report": report_1,
            "history": history,
            "confession_enabled": bool(confession),
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # If confession is ON, always run pass 2 with majority evidence
    messages = build_majority_messages(majority_neighbors, report_1, pass_id=2)
    report_2 = generate_report_base(
        model=model,
        processor=processor,
        image_path=image_path,
        prompt_messages=messages,
    )
    verify_2 = verify_true_false_lora(
        model=model,
        tokenizer=processor.tokenizer,
        verify_prompt=build_verify_prompt(report_2, neighbors, mode="majority"),
    ) if use_lora else None

    history.append({
        "pass": 2,
        "stage": "majority_revision",
        "report": report_2,
        "verify": verify_2,
    })

    if STOP_IF_UNCHANGED and normalize_report_text(report_2) == normalize_report_text(report_1):
        return {
            "final_report": report_2,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Case 1: verify_1 TRUE -> end after pass 2
    if use_lora and verify_1 is not None and verify_1.get("pred") == "TRUE":
        return {
            "final_report": report_2,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Case 2: verify_1 FALSE -> inspect pass 2 verifier
    if use_lora and verify_2 is not None and verify_2.get("pred") == "TRUE":
        return {
            "final_report": report_2,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Pass 3: use minority text annotated as false
    messages = build_minority_false_messages(minority_neighbors, report_2, pass_id=3)
    report_3 = generate_report_base(
        model=model,
        processor=processor,
        image_path=image_path,
        prompt_messages=messages,
    )
    verify_3 = verify_true_false_lora(
        model=model,
        tokenizer=processor.tokenizer,
        verify_prompt=build_verify_prompt(report_3, neighbors, mode="minority_false"),
    ) if use_lora else None

    history.append({
        "pass": 3,
        "stage": "minority_false_revision",
        "report": report_3,
        "verify": verify_3,
    })

    if use_lora and verify_3 is not None and verify_3.get("pred") == "TRUE":
        return {
            "final_report": report_3,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Pass 4: final mixed pass, always keep as final answer
    messages = build_mixed_final_messages(majority_neighbors, minority_neighbors, report_3, pass_id=4)
    report_4 = generate_report_base(
        model=model,
        processor=processor,
        image_path=image_path,
        prompt_messages=messages,
    )
    verify_4 = verify_true_false_lora(
        model=model,
        tokenizer=processor.tokenizer,
        verify_prompt=build_verify_prompt(report_4, neighbors, mode="mixed"),
    ) if use_lora else None

    history.append({
        "pass": 4,
        "stage": "final_mixed_revision",
        "report": report_4,
        "verify": verify_4,
    })

    return {
        "final_report": report_4,
        "history": history,
        "confession_enabled": True,
        "passes_run": len(history),
        "majority_flag": majority_flag,
    }


In [10]:
# ============================================================
# 9) Batch prediction over one FAISS test set
# ============================================================

def batch_predict_phi4(
    *,
    test_index: faiss.Index,
    test_meta: List[Dict[str, Any]],
    gal_index: faiss.Index,
    gal_meta: List[Dict[str, Any]],
    use_lora: bool,
    confession: bool,
    n_items: Optional[int] = None,
    k: int = 10,
) -> List[Dict[str, Any]]:
    total = len(test_meta) if n_items is None else min(int(n_items), len(test_meta))
    outputs: List[Dict[str, Any]] = []

    for tid in range(total):
        row = test_meta[tid]
        qvec = query_vec_from_test_id(test_index, tid)
        neighbors = retrieve_gallery_neighbors(gal_index, gal_meta, qvec, k=k)

        pred = rag_generate_phi4_report(
            model=verifier_base,
            processor=processor,
            image_path=row["image_path"],
            neighbors=neighbors,
            k=k,
            use_lora=use_lora,
            confession=confession,
        )

        out = {
            "test_id": tid,
            "faiss_id": row.get("faiss_id", tid),
            "image_path": row.get("image_path"),
            "gt_report": build_gt_report_text(row),
            "pred_report": pred["final_report"],
            "history": pred["history"],
            "confession_enabled": pred["confession_enabled"],
            "passes_run": pred["passes_run"],
            "majority_flag": pred.get("majority_flag"),
            "neighbors": [
                {
                    "rank": n.get("_rank"),
                    "score": n.get("_score"),
                    "normal": _row_normal(n),
                    "labels": _row_labels(n),
                    "caption": _row_caption(n),
                    "FINDINGS": _row_findings(n),
                    "IMPRESSION": _row_impression(n),
                }
                for n in neighbors
            ],
        }
        outputs.append(out)

        if (tid + 1) % int(PROGRESS_EVERY) == 0:
            print(f"Processed {tid + 1}/{total}")

    return outputs


In [11]:

# ============================================================
# 10) Text-only evaluation metrics for final report text
# ============================================================
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore


def _tok_text(s: str) -> List[str]:
    s = (s or "").strip().lower()
    return s.split() if s else []


def compute_text_metrics(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    smooth = SmoothingFunction().method1
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    bleu_scores = []
    r1_scores = []
    r2_scores = []
    rl_scores = []

    refs = []
    hyps = []

    for r in records:
        ref = (r.get("gt_report") or "").strip()
        hyp = (r.get("pred_report") or "").strip()
        if not ref or not hyp:
            continue

        refs.append(ref)
        hyps.append(hyp)

        bleu_scores.append(sentence_bleu([_tok_text(ref)], _tok_text(hyp), smoothing_function=smooth))
        rs = rouge.score(ref, hyp)
        r1_scores.append(rs["rouge1"].fmeasure)
        r2_scores.append(rs["rouge2"].fmeasure)
        rl_scores.append(rs["rougeL"].fmeasure)

    summary = {
        "N_text_eval": len(refs),
        "BLEU_mean": float(np.mean(bleu_scores)) if bleu_scores else float("nan"),
        "ROUGE1_F": float(np.mean(r1_scores)) if r1_scores else float("nan"),
        "ROUGE2_F": float(np.mean(r2_scores)) if r2_scores else float("nan"),
        "ROUGEL_F": float(np.mean(rl_scores)) if rl_scores else float("nan"),
    }

    if refs and hyps:
        P, R, F = bertscore(hyps, refs, lang="en", verbose=False)
        summary["BERTScore_P"] = float(P.mean().item())
        summary["BERTScore_R"] = float(R.mean().item())
        summary["BERTScore_F1"] = float(F.mean().item())
    else:
        summary["BERTScore_P"] = float("nan")
        summary["BERTScore_R"] = float("nan")
        summary["BERTScore_F1"] = float("nan")

    return summary


In [12]:
# ================================================================================================
# 10: Compute all metrics: text similarity + label accuracy + hallucination + RadGraph
# ================================================================================================

import re
import math
import numpy as np
from typing import Dict, Any, List
from collections import Counter

from sklearn.metrics import f1_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

# Optional CIDEr
try:
    from pycocoevalcap.cider.cider import Cider
    _HAS_CIDER = True
except Exception:
    _HAS_CIDER = False


LABELS_13 = [
    "atelectasis",
    "cardiomegaly",
    "consolidation",
    "edema",
    "enlarged cardiomediastinum",
    "fracture",
    "lung lesion",
    "lung opacity",
    "pleural effusion",
    "pleural other",
    "pneumonia",
    "pneumothorax",
    "support devices",
]

_NO_DETAIL_STRINGS = {
    "",
    "no detailed information",
    "none",
    "n/a",
    "na",
    "unknown",
}


def _clean_text(x: Any) -> str:
    x = "" if x is None else str(x)
    return re.sub(r"\s+", " ", x).strip()


def _safe_lower(x: Any) -> str:
    return _clean_text(x).lower()


def extract_labels_from_text(text: str) -> Dict[str, Any]:
    """
    Lightweight label extraction from report text.
    Uses exact label phrase matching.
    """
    text_low = _safe_lower(text)
    labels = []

    for lab in LABELS_13:
        if re.search(rf"\b{re.escape(lab)}\b", text_low):
            labels.append(lab)

    normal = "yes" if len(labels) == 0 else "no"
    return {"labels": labels, "normal": normal}


def build_label_vectors(records: List[Dict[str, Any]]):
    y_true = []
    y_pred = []
    normal_true = []
    normal_pred = []

    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))

        gt_vec = [1 if l in gt["labels"] else 0 for l in LABELS_13]
        pr_vec = [1 if l in pr["labels"] else 0 for l in LABELS_13]

        y_true.append(gt_vec)
        y_pred.append(pr_vec)

        normal_true.append(1 if gt["normal"] == "yes" else 0)
        normal_pred.append(1 if pr["normal"] == "yes" else 0)

    return (
        np.array(y_true, dtype=int),
        np.array(y_pred, dtype=int),
        np.array(normal_true, dtype=int),
        np.array(normal_pred, dtype=int),
    )


def compute_generation_accuracy(records: List[Dict[str, Any]]) -> Dict[str, float]:
    y_true, y_pred, normal_true, normal_pred = build_label_vectors(records)

    if len(records) == 0:
        return {
            "N_text_eval": 0,
            "MajorityNormalAcc": float("nan"),
            "NormalAccuracy": float("nan"),
            "MacroF1": float("nan"),
            "MicroF1": float("nan"),
            "HammingAcc": float("nan"),
        }

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    hamming_acc = float((y_true == y_pred).mean())
    normal_acc = float((normal_true == normal_pred).mean())

    majority_label = 1 if normal_true.mean() >= 0.5 else 0
    majority_normal_acc = float((normal_pred == majority_label).mean())

    return {
        "N_text_eval": int(len(records)),
        "MajorityNormalAcc": majority_normal_acc,
        "NormalAccuracy": normal_acc,
        "MacroF1": float(macro_f1),
        "MicroF1": float(micro_f1),
        "HammingAcc": hamming_acc,
    }


def compute_fer(records: List[Dict[str, Any]]) -> float:
    total_pred = 0
    false_pred = 0

    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        gt_set = set(gt["labels"])

        for lab in pr["labels"]:
            total_pred += 1
            if lab not in gt_set:
                false_pred += 1

    return false_pred / max(1, total_pred)


def compute_fer_no_normal(records: List[Dict[str, Any]]) -> float:
    total_pred = 0
    false_pred = 0

    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))

        if gt["normal"] == "yes":
            continue

        gt_set = set(gt["labels"])
        for lab in pr["labels"]:
            total_pred += 1
            if lab not in gt_set:
                false_pred += 1

    return false_pred / max(1, total_pred)


def compute_omission_rate(records: List[Dict[str, Any]]) -> float:
    total_gt = 0
    missed = 0

    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        pr_set = set(pr["labels"])

        for lab in gt["labels"]:
            total_gt += 1
            if lab not in pr_set:
                missed += 1

    return missed / max(1, total_gt)


def compute_coverage(records: List[Dict[str, Any]]) -> float:
    vals = []

    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))

        if len(gt["labels"]) == 0:
            continue

        hit = sum(1 for lab in gt["labels"] if lab in pr["labels"])
        vals.append(hit / len(gt["labels"]))

    return float(np.mean(vals)) if vals else 0.0


def compute_bleu_mean(records: List[Dict[str, Any]]) -> float:
    smoothie = SmoothingFunction().method1
    vals = []

    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))

        if not ref:
            continue

        ref_tokens = ref.split()
        hyp_tokens = hyp.split() if hyp else []

        vals.append(
            sentence_bleu(
                [ref_tokens],
                hyp_tokens,
                smoothing_function=smoothie,
            )
        )

    return float(np.mean(vals)) if vals else float("nan")


def compute_rouge_scores(records: List[Dict[str, Any]]) -> Dict[str, float]:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []

    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))

        if not ref:
            continue

        s = scorer.score(ref, hyp)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)

    return {
        "ROUGE1_F": float(np.mean(r1)) if r1 else float("nan"),
        "ROUGE2_F": float(np.mean(r2)) if r2 else float("nan"),
        "ROUGEL_F": float(np.mean(rl)) if rl else float("nan"),
    }


def compute_bertscore(records: List[Dict[str, Any]]) -> Dict[str, float]:
    refs, hyps = [], []

    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))
        if ref:
            refs.append(ref)
            hyps.append(hyp)

    if not refs:
        return {
            "BERTScore_P": float("nan"),
            "BERTScore_R": float("nan"),
            "BERTScore_F1": float("nan"),
        }

    P, R, F1 = bertscore_score(hyps, refs, lang="en", verbose=False)
    return {
        "BERTScore_P": float(P.mean().item()),
        "BERTScore_R": float(R.mean().item()),
        "BERTScore_F1": float(F1.mean().item()),
    }


def compute_cider(records: List[Dict[str, Any]]) -> float:
    if not _HAS_CIDER:
        return float("nan")

    gts = {}
    res = {}
    kept = 0

    for i, r in enumerate(records):
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))
        if not ref:
            continue
        gts[kept] = [ref]
        res[kept] = [hyp]
        kept += 1

    if kept == 0:
        return float("nan")

    scorer = Cider()
    score, _ = scorer.compute_score(gts, res)
    return float(score)


def compute_radgraph_f1(records: List[Dict[str, Any]], model_type: str = "radgraph-xl") -> Dict[str, float]:
    """
    Requires the compatibility patch before importing radgraph:
        import transformers
        from torch.optim import AdamW as TorchAdamW
        if not hasattr(transformers, "AdamW"):
            transformers.AdamW = TorchAdamW
    """
    from radgraph import F1RadGraph

    refs = [str(r.get("gt_report", "")).strip() for r in records]
    hyps = [str(r.get("pred_report", "")).strip() for r in records]

    pairs = [(ref, hyp) for ref, hyp in zip(refs, hyps) if ref and hyp]
    if not pairs:
        return {
            "RadGraph_E": float("nan"),
            "RadGraph_ER": float("nan"),
            "RadGraph_BER": float("nan"),
        }

    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]

    scorer = F1RadGraph(reward_level="all", model_type=model_type)
    mean_reward, reward_list, hypothesis_annotation_lists, reference_annotation_lists = scorer(
        hyps=hyps,
        refs=refs,
    )

    rg_e, rg_er, rg_bar_er = mean_reward
    return {
        "RadGraph_E": float(rg_e),
        "RadGraph_ER": float(rg_er),
        "RadGraph_BER": float(rg_bar_er),
    }


def compute_full_metrics(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Master metrics function for one JSONL result set.
    """
    metrics = {}

    # Accuracy / label metrics
    metrics.update(compute_generation_accuracy(records))

    # Hallucination / omission
    metrics["FER"] = compute_fer(records)
    metrics["FER_no_normal"] = compute_fer_no_normal(records)
    metrics["OmissionRate"] = compute_omission_rate(records)
    metrics["Coverage"] = compute_coverage(records)

    # Text similarity
    metrics["BLEU_mean"] = compute_bleu_mean(records)
    metrics.update(compute_rouge_scores(records))
    metrics.update(compute_bertscore(records))
    metrics["CIDEr"] = compute_cider(records)

    # Diagnostics
    num_records = len(records)
    num_empty = sum(int(not _clean_text(r.get("pred_report", ""))) for r in records)
    avg_pred_len = (
        sum(len(_clean_text(r.get("pred_report", ""))) for r in records) / max(1, num_records)
    )

    metrics["EmptyPredictionRate"] = num_empty / max(1, num_records)
    metrics["AvgPredictionLength"] = float(avg_pred_len)

    # RadGraph (optional but recommended)
    try:
        rg = compute_radgraph_f1(records, model_type="radgraph-xl")
        metrics.update(rg)
        metrics["RadGraphF1"] = metrics["RadGraph_ER"]
        metrics["RadGraph_available"] = True
    except Exception as e:
        metrics["RadGraph_available"] = False
        metrics["RadGraph_error"] = repr(e)

    return metrics

In [13]:
# ============================================================
# 12) Helper: rebuild combined comparison CSV from saved 12 outputs
# ============================================================

from pathlib import Path
import json
import csv
from typing import Any, Dict, List


def load_jsonl_records(jsonl_path: Path) -> List[Dict[str, Any]]:
    records = []
    if not jsonl_path.exists():
        return records
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


# Per-experiment filename helpers — unique across all 4 methods
def _exp_jsonl_name(exp_name: str) -> str:
    return f"phi4_{exp_name}_eval_records.jsonl"


def _exp_summary_name(exp_name: str) -> str:
    return f"phi4_{exp_name}_summary.csv"


def attach_run_metadata_to_summary(
    summary: Dict[str, Any],
    dataset_name: str,
    dataset_root: Path,
    exp: Dict[str, Any],
    records: List[Dict[str, Any]],
) -> Dict[str, Any]:
    summary = dict(summary)

    use_lora       = bool(exp["USE_LORA"])
    use_confession = bool(exp["USE_CONFESSION"])

    summary["Confession"]   = use_confession
    summary["LoRA_enabled"] = use_lora
    summary["TopK"]         = int(TOP_K)
    summary["Model"]        = MODEL_ID
    summary["LoRA"]         = str(LORA_DIR) if use_lora else "OFF"
    summary["Dataset"]      = dataset_name
    summary["DatasetRoot"]  = str(dataset_root)
    summary["N_records"]    = len(records)
    summary["EmptyPredictionRate"] = (
        sum(int(not str(r.get("pred_report", "")).strip()) for r in records) / max(1, len(records))
    )
    return summary


def build_comparison_rows_from_payloads(all_results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    comparison_rows = []

    for payload in all_results:
        s = payload["summary"]

        row = {
            "Dataset": payload["dataset"],
            "Experiment": payload["experiment"],
            "SaveDir": payload["save_dir"],
            "USE_LORA": s.get("LoRA_enabled"),
            "USE_CONFESSION": s.get("Confession"),
            "TopK": s.get("TopK"),
            "N_records": s.get("N_records"),
            "N_text_eval": s.get("N_text_eval"),
            "BLEU_mean": s.get("BLEU_mean"),
            "ROUGE1_F": s.get("ROUGE1_F"),
            "ROUGE2_F": s.get("ROUGE2_F"),
            "ROUGEL_F": s.get("ROUGEL_F"),
            "BERTScore_P": s.get("BERTScore_P"),
            "BERTScore_R": s.get("BERTScore_R"),
            "BERTScore_F1": s.get("BERTScore_F1"),
            "CIDEr": s.get("CIDEr"),
            "Coverage": s.get("Coverage"),
            "EmptyPredictionRate": s.get("EmptyPredictionRate"),
            "AvgPredictionLength": s.get("AvgPredictionLength"),
            "RuntimeSec_total": s.get("RuntimeSec_total"),
            "RuntimeSec_per_study": s.get("RuntimeSec_per_study"),
        }

        for key in [
            "MajorityNormalAcc", "NormalAccuracy", "MacroF1", "MicroF1", "HammingAcc",
            "FER", "FER_no_normal", "OmissionRate",
            "RadGraph_E", "RadGraph_ER", "RadGraph_BER", "RadGraphF1", "RadGraph_available",
        ]:
            if key in s:
                row[key] = s.get(key)

        comparison_rows.append(row)

    return comparison_rows


def save_comparison_csv(comparison_rows: List[Dict[str, Any]], out_csv: Path):
    if not comparison_rows:
        print("⚠️ No comparison rows to save.")
        return

    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(comparison_rows[0].keys()))
        writer.writeheader()
        for row in comparison_rows:
            writer.writerow(row)

    print("✅ Saved combined comparison CSV:", out_csv)


def rebuild_all_results_from_saved_outputs(
    dataset_roots: Dict[str, Path],
    experiments: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Rebuild ALL_RESULTS by reading saved JSONL outputs from disk
    and recomputing compute_full_metrics(records) for each of the 12 runs.
    """
    rebuilt_results = []

    for dataset_name, dataset_root in dataset_roots.items():
        for exp in experiments:
            exp_name     = exp["name"]
            save_dir     = dataset_root / exp["subdir"]
            result_jsonl = save_dir / _exp_jsonl_name(exp_name)
            summary_csv  = save_dir / _exp_summary_name(exp_name)

            if not result_jsonl.exists():
                print(f"⚠️ Missing JSONL, skipping: {result_jsonl}")
                continue

            records = load_jsonl_records(result_jsonl)
            summary = compute_full_metrics(records)
            summary = attach_run_metadata_to_summary(
                summary=summary,
                dataset_name=dataset_name,
                dataset_root=dataset_root,
                exp=exp,
                records=records,
            )

            with open(summary_csv, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["metric", "value"])
                for k, v in summary.items():
                    w.writerow([k, v])

            print("✅ Refreshed summary:", summary_csv)

            rebuilt_results.append({
                "dataset"     : dataset_name,
                "experiment"  : exp_name,
                "save_dir"    : str(save_dir),
                "result_jsonl": str(result_jsonl),
                "summary_csv" : str(summary_csv),
                "summary"     : summary,
            })

    return rebuilt_results


In [14]:
# ============================================================
# 13) Run the 12 evaluation combinations + save JSONL/CSV
#     3 datasets × 4 methods = 12 combinations
# ============================================================

def save_records_and_summary(
    records: List[Dict[str, Any]],
    summary: Dict[str, Any],
    save_dir: Path,
    exp_name: str,
):
    save_dir.mkdir(parents=True, exist_ok=True)

    result_jsonl = save_dir / _exp_jsonl_name(exp_name)
    summary_csv  = save_dir / _exp_summary_name(exp_name)

    with open(result_jsonl, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("✅ Saved records:", result_jsonl)

    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])
        for k, v in summary.items():
            w.writerow([k, v])
    print("✅ Saved summary:", summary_csv)

    return result_jsonl, summary_csv


def compute_summary(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    if "compute_full_metrics" in globals():
        return compute_full_metrics(records)
    return compute_text_metrics(records)


def run_one_experiment(dataset_name: str, dataset_root: Path, exp: Dict[str, Any]) -> Dict[str, Any]:
    use_lora       = bool(exp["USE_LORA"])
    use_confession = bool(exp["USE_CONFESSION"])
    exp_name       = exp["name"]
    subdir         = exp["subdir"]

    test_index, test_meta, gal_index, gal_meta = load_dataset_bundles(dataset_root)
    save_dir = dataset_root / subdir
    save_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print("DATASET    :", dataset_name)
    print("TEST ROOT  :", dataset_root)
    print("SAVE DIR   :", save_dir)
    print("USE_LORA   :", use_lora)
    print("CONFESSION :", use_confession)
    print("=" * 80)

    start_time = time.time()
    records = batch_predict_phi4(
        test_index=test_index,
        test_meta=test_meta,
        gal_index=gal_index,
        gal_meta=gal_meta,
        use_lora=use_lora,
        confession=use_confession,
        n_items=N_ITEMS,
        k=TOP_K,
    )
    elapsed = time.time() - start_time

    summary = compute_summary(records)
    summary["RuntimeSec_total"]     = float(elapsed)
    summary["RuntimeSec_per_study"] = float(elapsed / max(1, len(records)))
    summary["Confession"]           = use_confession
    summary["LoRA_enabled"]         = use_lora
    summary["TopK"]                 = int(TOP_K)
    summary["Model"]                = MODEL_ID
    summary["LoRA"]                 = str(LORA_DIR) if use_lora else "OFF"
    summary["Dataset"]              = dataset_name
    summary["DatasetRoot"]          = str(dataset_root)
    summary["N_records"]            = len(records)
    summary["EmptyPredictionRate"]  = (
        sum(int(not str(r.get("pred_report", "")).strip()) for r in records)
        / max(1, len(records))
    )

    result_jsonl, summary_csv = save_records_and_summary(records, summary, save_dir, exp_name)

    return {
        "dataset"     : dataset_name,
        "experiment"  : exp_name,
        "save_dir"    : str(save_dir),
        "result_jsonl": str(result_jsonl),
        "summary_csv" : str(summary_csv),
        "summary"     : summary,
    }


ALL_RESULTS: List[Dict[str, Any]] = []

total_combos = len(DATASET_ROOTS) * len(EXPERIMENTS)
combo_idx    = 0
for dataset_name, dataset_root in DATASET_ROOTS.items():
    for exp in EXPERIMENTS:
        combo_idx += 1
        print(f"\n[{combo_idx}/{total_combos}] dataset={dataset_name}  method={exp['name']}")
        run_payload = run_one_experiment(dataset_name, dataset_root, exp)
        ALL_RESULTS.append(run_payload)

comparison_rows = build_comparison_rows_from_payloads(ALL_RESULTS)
save_comparison_csv(comparison_rows, COMPARISON_CSV)

print("\n================ 12-COMBO RUN SUMMARY ================")
for payload in ALL_RESULTS:
    print(payload["dataset"], "|", payload["experiment"], "|", payload["summary_csv"])


✅ Loaded dataset bundles for faiss_val_mimic_biomedclip
test size: 634 gallery size: 10905

DATASET: mimic
TEST ROOT: /data/liangz2/openi/faiss_val_mimic_biomedclip
SAVE DIR: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_Noverify_Noconfess
USE_LORA: False
USE_CONFESSION: False
Processed 50/634
Processed 100/634
Processed 150/634
Processed 200/634
Processed 250/634
Processed 300/634
Processed 350/634
Processed 400/634
Processed 450/634
Processed 500/634
Processed 550/634
Processed 600/634


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_Noverify_Noconfess/phi4_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_Noverify_Noconfess/phi4_summary.csv
✅ Loaded dataset bundles for faiss_val_mimic_biomedclip
test size: 634 gallery size: 10905

DATASET: mimic
TEST ROOT: /data/liangz2/openi/faiss_val_mimic_biomedclip
SAVE DIR: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_verify_Noconfess
USE_LORA: True
USE_CONFESSION: False
Processed 50/634
Processed 100/634
Processed 150/634
Processed 200/634
Processed 250/634
Processed 300/634
Processed 350/634
Processed 400/634
Processed 450/634
Processed 500/634
Processed 550/634
Processed 600/634


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_verify_Noconfess/phi4_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_verify_Noconfess/phi4_summary.csv
✅ Loaded dataset bundles for faiss_val_mimic_biomedclip
test size: 634 gallery size: 10905

DATASET: mimic
TEST ROOT: /data/liangz2/openi/faiss_val_mimic_biomedclip
SAVE DIR: /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_verify_confess
USE_LORA: True
USE_CONFESSION: True
Processed 50/634
Processed 100/634
Processed 150/634
Processed 200/634
Processed 250/634
Processed 300/634
Processed 350/634
Processed 400/634
Processed 450/634
Processed 500/634
Processed 550/634
Processed 600/634


...

Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_Noverify_Noconfess/phi4_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_Noverify_Noconfess/phi4_summary.csv
✅ Loaded dataset bundles for faiss_val_combined_biomedclip
test size: 1034 gallery size: 14589

DATASET: combined
TEST ROOT: /data/liangz2/openi/faiss_val_combined_biomedclip
SAVE DIR: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_verify_Noconfess
USE_LORA: True
USE_CONFESSION: False
Processed 50/1034
Processed 100/1034
Processed 150/1034
Processed 200/1034
Processed 250/1034
Processed 300/1034
Processed 350/1034
Processed 400/1034
Processed 450/1034
Processed 500/1034
Processed 550/1034
Processed 600/1034
Processed 650/1034
Processed 700/1034
Processed 750/1034
Processed 800/1034
Processed 850/1034
Processed 900/1034
Processed 950/1034
Processed 1000/1034


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_verify_Noconfess/phi4_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_verify_Noconfess/phi4_summary.csv
✅ Loaded dataset bundles for faiss_val_combined_biomedclip
test size: 1034 gallery size: 14589

DATASET: combined
TEST ROOT: /data/liangz2/openi/faiss_val_combined_biomedclip
SAVE DIR: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_verify_confess
USE_LORA: True
USE_CONFESSION: True
Processed 50/1034
Processed 100/1034
Processed 150/1034
Processed 200/1034
Processed 250/1034
Processed 300/1034
Processed 350/1034
Processed 400/1034
Processed 450/1034
Processed 500/1034
Processed 550/1034
Processed 600/1034
Processed 650/1034
Processed 700/1034
Processed 750/1034
Processed 800/1034
Processed 850/1034
Processed 900/1034
Processed 950/1034
Processed 1000/1034


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_verify_confess/phi4_confession_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_combined_biomedclip/phi4_reason_verify_confess/phi4_confession_summary.csv
✅ Saved combined comparison CSV: /data/liangz2/openi/phi4_rag_eval_9combo/phi4_9combo_comparison.csv

================ 9-COMBO RUN SUMMARY ================
mimic | phi4_reason_Noverify_Noconfess | /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_Noverify_Noconfess/phi4_summary.csv
mimic | phi4_reason_verify_Noconfess | /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_verify_Noconfess/phi4_summary.csv
mimic | phi4_reason_verify_confess | /data/liangz2/openi/faiss_val_mimic_biomedclip/phi4_reason_verify_confess/phi4_confession_summary.csv
openi | phi4_reason_Noverify_Noconfess | /data/liangz2/openi/faiss_val_openi_biomedclip/phi4_reason_Noverify_Noconfess/phi4_summary.csv
openi | phi4_reason_verify_No

In [ ]:

# ============================================================
# 12) Quick inspection of one saved prediction from the last run
# ============================================================
if ALL_RESULTS:
    last_result_jsonl = Path(ALL_RESULTS[-1]["result_jsonl"])
    if last_result_jsonl.exists():
        with open(last_result_jsonl, "r", encoding="utf-8") as f:
            first_line = next((ln for ln in f if ln.strip()), None)
        if first_line:
            ex = json.loads(first_line)
            print("result_jsonl:", last_result_jsonl)
            print("image_path:", ex["image_path"])
            print("\n--- GT REPORT ---\n")
            print(ex["gt_report"])
            print("\n--- FINAL PRED REPORT ---\n")
            print(ex["pred_report"])
            print("\n--- HISTORY ---\n")
            for h in ex["history"]:
                print(f"[pass {h['pass']}] stage={h.get('stage')} verify={h.get('verify')}\n{h['report']}\n")
